# Practical 3: Modelling the Zwalm catchment with a conceptual rainfall-runoff model
[Lucas Boeykens](mailto:lucas.boeykens@kuleuven.be), [Elise Huysman](mailto:elyse.huysman@kuleuven.be) - 2026-12-17

## Hydrological models
In this practical, you will experiment with a hydrological model to simulate the discharge in the Zwalm catchment (East Flanders). Hydrological models typically represent a catchment as a system, with certain in- and outputs (Fig 1). A specific subset of these models, rainfall-runoff models, typically have precipitation (P) and evapotranspiration (E) as the input and output discharge (Q), the latter representing the discharge at the outlet of the catchment of interest. Today, you will work with a rainfall-runoff model that is a simplified and modified version of the FLEX model as described in [Fenicia et al. (2026)](https://doi.org/10.5194/hess-10-139-2006) and [Fenicia et al. (2008)](https://doi.org/10.1029/2006WR005563). First, you will learn how to use and calibrate the model, and afterwards you will use the calibrated model to simulate Q within the Zwalm catchment.

<div style="text-align: center;">
    <img src="../../img/Representation_hydrological_model.png" width="750" height="350">
    <div style="text-align: left;"><figcaption><b>Figure 1:</b> Catchment representation within a hydrological model.The catchment is represented by a system with inputs, state variables, parameters and an output. For rainfall-runoff models specifically, the inputs are precipitation (P) and evapotranspiration (E), whereas the output is discharge (Q) at the outlet of the catchment.</figcaption></div>
</div>

## Description of the FLEX model
The FLEX model ([Fenicia et al. (2026)](https://doi.org/10.5194/hess-10-139-2006) and [Fenicia et al. (2008)](https://doi.org/10.1029/2006WR005563); Fig 2) is a lumped rainfall-runoff model. As such, it treats the entire catchment as a single hydrological unit, using catchement-averaged inputs. The model considers three state variables, corresponding to the volume of water in three reservoirs:
1. The unsaturated soil reservoir $S(t)$ [mm]
2. The slow reacting reservoir $S_1(t)$ [mm], conceptually representing the groundwater system
3. The fast reacting reservoir $S_2(t)$ [mm]

Besides these states, the model has 10 different parameters that can be calibrated using observed discharges for a specific catchment (e.g., the Zwalm catchment).

<div style="text-align: center;">
    <img src="../../img/FLEX_schets.png" width="350" height="400">
    <div style="text-align: left;"><figcaption><b>Figure 2:</b> Schematical representation of the FLEX model. The model has 3 state variables and 10 parameters to be calibrated.</figcaption></div>
</div>

The model is forced with precipitation $P(t)$ and potential evaporation $E_p(t)$^[$\text{mm is used as a unit here, as it allows to implement the model independently of the catchment area}: \mathrm{ mm} = 1 \cdot 10^{-3} \mathrm{ m}^3 / \mathrm{m}^2$]. In a first step, FLEX splits the net precipitation into two components: $P_{in}$, the input to to the unsaturated soil reservoir, and $P_{exc}$, which corresponds to the excess precipitation ( $=P-P_{in}$ ) or the part of the precipitation that cannot be retained by the soil. Hereby, the fraction of $P$ that can infiltrate in the soil reservoir depends on the water volume present in the soil reservoir at a specific moment ($S_t$), and the amount of actual evaporation ($E_a$) occuring at a specific time step. The exces precipitation is then further split into two components $(P_1$ and $P_2)$, dependent on $S(t)$ and a parameter $\alpha$. The more water is present in the soil reservoir, the more of $P_{exc}$ will be distributed to the fast reacting reservoir, and vice versa for the slow reacting reservoir. 

The slow reacting reservoir ($S_1$) is fed by two sources: $P_1$ and the percolation from the soil reservoir $(Q_p)$. This reservoir is considered to be linear, such that the resulting discharge $Q_1 = \kappa_{1}S_1(t)$, with $\kappa_{1}$ being a dimensionless parameter representing the 'reciprocal of the residence time for the slow reacting reservoir'. The second reservoir, on the other hand, is non-linear. As a result, the discharge produced by this reservoir, $Q_2$, depends on the the volume of water in this state variable and grows exponentially with increasing $S_2(t)$-values (Fig 3).

<div style="text-align: center;">
    <img src="../../img/example_Q2.png" width="300" height="200">
    <div style="text-align: left;">
        <figcaption><strong>Figure 3:</strong> Outflow discharge from the fast reacting reservoir in function of the volume of water stored in this reservoir.
    </figcaption>
</div>
</div>

## Running the FLEX model within python
The code to run the model can be found in the python-script `Scripts/scripts_hydrological_model/model.py`. In order to run the model, you must first instantiate multiple classes. A [class](https://docs.python.org/3/tutorial/classes.html) is a blueprint to create an object that bundles attributes (data or inputs) and functionalities (methods or functions). To create an object (from which you can use the functions), you must first create an instance (object) after which you can call its attributes and its methods (functions). 

The necessary classes to instantiate are:
- `States`: this class contains the initial states of the three water reservoirs. It thus contains information of the amount of water that is present in $S$, $S_1$ and $S_2$ at time $t=0$. 
- `Parameters`: a class to create an object containing the parameters of the model. Together with the states, it allows to make an instance of the FLEX model to perform a model run. 
- `RainfallRunoffModel`: this class contains the implementation of the above described model. It uses the states and parameters classes and allows to do a model simulation.
- `ForcingsTimeSeries`: a class which has the forcing data ($P$ and $E$) as attributes to perform a model run. 
- `Simulation`: allows to run a simulation with the `RainFallRunoffModel` and the `ForcingsTimeSeries` for a specified time range. It internally calls the `euler_step` functionality of the `RainfallRunoffModel` class for the different time steps within the simulation period.

The code below gives an example on how to make instances of the above described classes and how to use them to run the FLEX model for a specified time period.


In [ ]:
# --- import modules ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys, rootutils
from pathlib import Path

ROOT_PATH = rootutils.find_root(search_from=Path.cwd(), indicator=["environment.yml", "Data", "Scripts"])
sys.path.append(str(ROOT_PATH))
ROOTDIR_DATA = os.path.join(ROOT_PATH, "Data", "processed")
sys.path.append(ROOTDIR_DATA)

from Scripts.scripts_hydrological_model.model import (
    ForcingsTimeSeries,
    Parameters,
    States,
    RainfallRunoffModel,
    Simulation
)
from Scripts.scripts_hydrological_model.calibration import (
    NelderMeadCalibration
)
from Scripts.constants import (
    CATCHMENT_AREA_ZWALM
)

First, we make an instance (object) of the initial states and parameters. Note that you can access the default attributes by instantiating the object with no arguments.

In [ ]:
# --- initialize the parameters ---
# Instantiate parameters with default values
parameters = Parameters() #NOTE: by not passing any arguments, the default values of the parameters will be used
print(f"Initial parameters: {parameters}")

# Change the S_max parameter to 400
parameters_update=Parameters(S_max=400) #NOTE: by passing the argument S_max=400, the S_max parameter will be updated to 400, while the other parameters will remain at their default values
print(f"Updated parameters: {parameters_update}")

# Access the S_max parameter of the parameters object
print(f"\nS_max parameter: {parameters.S_max}")

# --- initialize the states ---
# initialize the states of the model
states = States()
print(f"\nInitial states: {states}")

# change the initial storage to half of the maximum storage (S_max/2)
states_update=States(S=parameters.S_max/2) #NOTE: we accessed the S_max parameter of the parameters object to set the initial storage to half of the maximum storage
print(f"\nUpdated states: {states_update}")

Next, we can use the states and parameters to make a RainfallRunoffModel object. As it is a lumped model using fluxes in mm, we must pass the area of the Zwalm catchment to make the fluxes specific to our catchment!

In [ ]:
# --- create the rainfall-runoff model object ---
# Create the object
model = RainfallRunoffModel(
    catchment_area=CATCHMENT_AREA_ZWALM,
    states=states,
    parameters=parameters,
)

# Check some of the attributes of the object: the parameters and states
print(f"Model parameters: {model.parameters}")
print(f"\nModel states: {model.states}")

Finally, we need the `ForcingTimeSeries` class to generate the forcings object. With the attributes of this data we can make a plot of the precipitation in time

In [ ]:
# --- create the time series object ---
# Read in the forcing data
df_forcings = pd.read_csv(
    os.path.join(ROOTDIR_DATA, "data_zwalm", "forcings_discharge.csv"),
    index_col=0,
    parse_dates=True
)

# Create the ForcingsTimeSeries object
forcings = ForcingsTimeSeries(
    precipitation=df_forcings["precipitation"].values,
    potential_evaporation=df_forcings["potential_evapotranspiration"].values,
    time=df_forcings.index.values
)

# Plot the precipitation
fig,ax=plt.subplots(figsize=(10, 6))
ax.plot(
    forcings.time,
    forcings.precipitation,
    color="#ADCF14",
    lw=1.6,
)
ax.spines["top"].set_visible(False)
ax.set_ylabel("Precipitation (mm)", fontsize=18)
ax.set_xlabel("Date [-]", fontsize=18)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=15)
plt.show()

Finally, we can generate the simulations using the `simulations` class and its `.run()` functionality. 

In [ ]:
# --- make a simulation ---
date_ini=pd.Timestamp("2018-01-01")
date_end=pd.Timestamp("2019-12-31")
n_days_warmup=50

# Adjust the initial date to account for the warm-up period
date_ini_warmup=date_ini-pd.Timedelta(days=n_days_warmup)
date_ini_warmup=date_ini_warmup if date_ini_warmup >= df_forcings.index.min() else df_forcings.index.min()

sim = Simulation(
    model=model,
    forcing_time_series=forcings,
    output_vars=["S", "S_1", "S_2", "Q_m3s", "Q_1", "Q_2"], #NOTE: you can change this list to have less outputs
    start_time=date_ini_warmup,
    end_time=date_end
)
sim.run()

df_sim=sim.output #NOTE: the output of the simulation is stored in the attribute "output"
isinstance(df_sim, pd.DataFrame) #NOTE: the output of the simulation is a pandas DataFrame

# --- plot the simulation results ---
# Leave out the warm-up period from the simulation results
df_sim=df_sim[df_sim.index >= date_ini]

# Plot the simulated discharge
fig,ax=plt.subplots(figsize=(12, 5))
ax.plot(
    df_sim.index,
    df_sim["Q_m3s"],
    color="tab:blue",
    lw=2,
)
ax.spines["top"].set_visible(False)
ax.set_ylabel("Discharge (m³ s$^{-1}$)", fontsize=18)
ax.set_xlabel("Date [-]", fontsize=18)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=15)
plt.show()

# Case study: simulating the discharge for the Zwalm catchment
## Data analysis of the meteorological forcing data
<div style="
    width: 100%;
    box-sizing: border-box;
    border: 1px solid #2979ff;
    border-left-width: 5px;
    background-color: #eef5ff;
    padding: 12px 16px;
    margin: 12px 0;
    border-radius: 4px;
">
<strong>Assignment 1</strong><br>
Inspect the precipitation and evaporation data for the time period 2018-01-01 2019-12-31. Answer the following questions.

1. Make a figure with the discharge (hydrograph) on the bottom and the precipitation (hyetograph) at the top of the figure. When do you observe higher precipitation periods: during summer or during winter? Does the discharge follow the same trends? What else do you think causes the observed discharge trends? Answer in max 2 sentences.  

2. During which time period (winter or summer) does Dunne runoff occur more? Give a possible explanation in max 3 sentences.

3. Read this [newsarticle](https://www.hln.be/lierde/onweders-blijven-watersnood-veroorzaken~a3da9cbc/). Make a plot of both the discharge and precipitation for the months of May and June. What was the maximal observed discharge after the precipitation event at the end of may (visual inspection is fine). Do you think this was Dunne or Horton runoff? Argue again in max 3 sentences.

4. Make a plot of the monthly evapotranspiration and discharge for the time period. When do you observe higher evaporation? Are higher or lower discharges observed during these periods? Answer in max 2 sentences.
</div>

<div class="alert alert-block alert-info"> <b>Tip!!</b> 

You can complete the pre coded functions below and use them to generate certain figures. Replace the brackets (""" """) with your own code.  </div>

In [ ]:
# --- make function that plots the hydrograph and hyetograph ---
def PlotHydroHyetographTimePeriod(
    df_forcings:pd.DataFrame,
    date_ini:pd.Timestamp = pd.Timestamp("2020-01-01"),
    date_end:pd.Timestamp = pd.Timestamp("2020-12-31"),
    discharge_column:str = "river_discharge",
    precipitation_column:str = "precipitation",
    ) -> plt.plot:
    """Function to plot a hydrograph and hyetograph for a given time period."""

    #---error handling---
    if not isinstance(df_forcings.index, pd.DatetimeIndex):
        raise ValueError("The index of df_forcings must be a DatetimeIndex.")

    #---select data for the time period---
    df_tp=df_forcings.loc[(df_forcings.index >= date_ini) & 
                        (df_forcings.index <= date_end)]

    #---make the plot---
    fig, ax = plt.subplots(figsize=(10, 6))

    #discharge at the bottom
    ax.plot(
        df_tp.index,
        df_tp[discharge_column],
        color="tab:blue",
        lw=2,
    )
    ax.set_ylabel("Discharge (m³ s$^{-1}$)", color="tab:blue", fontsize=18)
    ax.tick_params(axis="y", labelcolor="tab:blue")
    ax.spines["top"].set_visible(False)

    #precipitation  at the top
    ax2 = ax.twinx()
    ax2.bar(
        df_tp.index,
        df_tp[precipitation_column],
        width=0.9,
        color="tab:orange",
        alpha=0.6,
    )
    ax2.invert_yaxis()
    ax2.set_ylim(df_tp[precipitation_column].max() * 1.1, 0) #0 at the top

    # Move ticks and label to the top
    ax2.xaxis.set_visible(False)
    ax2.yaxis.set_label_position("right")
    ax2.yaxis.tick_right()

    ax2.set_ylabel("Precipitation (mm)", color="tab:orange", fontsize=18)
    ax2.tick_params(axis="y", labelcolor="tab:orange")

    # Cosmetics
    ax.grid(axis="y", alpha=0.3)
    ax.set_xlabel("Date [-]", fontsize=18)
    fig.tight_layout()

    ax.tick_params(axis="both", labelsize=15)
    ax2.tick_params(axis="y", labelsize=15)

    plt.show()

def PlotMonthlyAggregates(
    df_forcings:pd.DataFrame,
    date_ini:pd.Timestamp = pd.Timestamp("2020-01-01"),
    date_end:pd.Timestamp = pd.Timestamp("2020-12-31"),
    discharge_column:str = "river_discharge",
    evapotranspiration_column:str = "potential_evapotranspiration"
    ) -> plt.plot:
    """Function to plot monthly aggregates for a given time period."""
    #---error handling---
    if not isinstance(df_forcings.index, pd.DatetimeIndex):
        raise ValueError("The index of df_forcings must be a DatetimeIndex.")
    
    #---select data for the time period---
    df_tp=df_forcings.loc[(df_forcings.index >= date_ini) & 
                        (df_forcings.index <= date_end)]

    #---calculate monthly aggregates---
    monthlyAggregates=df_tp[[discharge_column, 
                             evapotranspiration_column]].\
                                groupby([df_tp.index.year,
                                         df_tp.index.month]).mean()
    #NOTE: you can use pandas.groupby("month").mean() to calculate the monthly mean

    #---make the plot---
    fig,ax=plt.subplots(figsize=(10, 6))

    monthlyAggregates[discharge_column].plot(
        kind="bar",
        color="tab:blue",
        ax=ax,
        width=0.8
    )
    
    ax2=ax.twinx()
    monthlyAggregates[evapotranspiration_column].plot(
        color="tab:orange",
        ax=ax2,
        lw=2
    )

    ax.tick_params(axis="both", labelsize=15)
    ax2.tick_params(axis="both", labelsize=15)
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", labelcolor="tab:blue")
    ax2.tick_params(axis="y", labelcolor="tab:orange")
    ax.set_ylabel(""" Set a proper label for the primary axis""", color="tab:blue", fontsize=18)
    ax2.set_ylabel(""" Set a proper label for the secondary axis""", color="tab:orange", fontsize=18)
    ax.set_xlabel("")

    fig.tight_layout()

    plt.show()


Question 1: 

Higher precipitation values are observed during winter months. Overall the discharge follows the same trends. It is also related to the seasonal cycle of evapotranspiration, which is lower during winter and higher during summer.

In [ ]:
PlotHydroHyetographTimePeriod(
    df_forcings=df_forcings,
    date_ini=pd.Timestamp("2018-01-01"),
    date_end=pd.Timestamp("2019-12-31")
)

Question 2: 

Dunne runoff occurs more during winter. Dunne runoff is characterised by saturated soils (also known as saturation excess overalnd flow). It occurs when the water table is high or the soil is saturated. Saturated soils are more often found in winter as there is less evaporation and (in general) more precipition.

Question 3: 
- max observed Q: > 2 m3/s. 
- Type of runoff: Horton runoff. It occured during the summer months when the soils are less saturated. Both events happened after large precipitation events, thus the water did not have the time to infiltrate in the soil

In [ ]:
PlotHydroHyetographTimePeriod(
    df_forcings=df_forcings,
    date_ini=pd.Timestamp("2018-05-01"),
    date_end=pd.Timestamp("2018-06-30")
)

Question 4:

Higher evaporation during summer months. Lower discharges are observed 

In [ ]:
PlotMonthlyAggregates(
    df_forcings=df_forcings,
    date_ini = pd.Timestamp("2018-01-01"),
    date_end = pd.Timestamp("2019-12-31")
    )

## Simulating with the FLEX model
<div style="
    width: 100%;
    box-sizing: border-box;
    border: 1px solid #2979ff;
    border-left-width: 5px;
    background-color: #eef5ff;
    padding: 12px 16px;
    margin: 12px 0;
    border-radius: 4px;
">
<strong>Assignment 2</strong><br>

Run the model for the time period of 2012-01-01 to 2019-12-31. Use the default states and parameters. Answer the following questions. 

1. Use 50 warm-up days to make the simualtion. Make a plot of the observed and simulated discharges.


2. Run the model again, but don't use any warm-up period. Make a plot of the simulated discharges of both runs for the first 6 months. Which model run (warm-up or no warm-up) displays the lowest discharge the first months? Why? Explain in max 5 sentences.

3. Compute the bias between the simulated and observed discharges for the model run with a warm-up period. When do you see the biggest deviations? In winter, or summer?


</div>

<div class="alert alert-block alert-info"> <b>Tip!!</b> 

You can use the function `runModelTimePeriod` to make a model simulation for a specific time period. You can import the function from: `Scripts.scripts.helper_functions`.  </div>

Question 1

In [ ]:
from Scripts.scripts_hydrological_model.helper_functions import (
    runModelTimePeriod
)

def PlotQobsSim(
        df_sim: pd.DataFrame,
        obs_column: str = "Qobs",
        sim_column: str = "Qsim",
        date_ini: pd.Timestamp | None = None,
        date_end: pd.Timestamp | None = None
    ) -> plt.plot:
    '''Function to plot observed and simulated discharge.'''
    if date_ini is not None and date_end is not None:
        df_sim = df_sim.loc[(df_sim.index >= date_ini) & 
                            (df_sim.index <= date_end)]

    # plot the simulated and observed discharge
    fig,ax=plt.subplots(figsize=(12, 5))
    ax.plot(
        df_sim.index,
        df_sim[sim_column],
        color="#6BD117",
        lw=2,
        alpha=1,
        label="Simulated discharge"
    )

    ax.plot(
        df_sim.index,
        df_sim[obs_column],
        color="#5495BB",
        lw=2,
        alpha=0.9,
        label="Observed discharge"
    )

    ax.legend(
        loc="upper left",
        fontsize=15,
        frameon=False
    )
    ax.spines["top"].set_visible(False)
    ax.set_ylabel("Discharge (m³ s$^{-1}$)", fontsize=18)
    ax.set_xlabel("Date [-]", fontsize=18)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=15)
    fig.tight_layout()

    plt.show()


# --- run the model for a given time period ---
parameters=Parameters()
states=States()
date_ini=pd.Timestamp("2012-01-01")
date_end=pd.Timestamp("2019-12-31")

df_sim=runModelTimePeriod(
    df_forcings=df_forcings,
    parameters=parameters, 
    states=states, 
    date_ini=date_ini, 
    date_end=date_end
    )

# --- plot the observed and simulated discharge ---
PlotQobsSim(
    df_sim=df_sim,
    )

Question 2:

- The model is initialized with the standard states, which are 0 for every reservoir. As such, the reservoirs still have to fill during the first modelin steps, and only produce a small discharge. When a warm-up period of 50 days (~2 months) is used, the reservoirs have already been partially filled, and thus the model will react differently to new forcings.

In [ ]:
df_sim_nwump=runModelTimePeriod(
    df_forcings=df_forcings,
    parameters=parameters, 
    states=states, 
    date_ini=date_ini, 
    date_end=date_end,
    n_warmup_days=0
    )

fig,ax=plt.subplots(figsize=(12, 5))
ax.plot(
    df_sim_nwump.index,
    df_sim_nwump["Qsim"],
    color="#6BD117",
    lw=2,
    alpha=1,
    label="Simulated discharge (no warm-up)"
)
ax.plot(
    df_sim.index,
    df_sim["Qsim"],
    color="#3C81AF",
    lw=2,
    alpha=1,
    label="Simulated discharge (warm-up)"
)
ax.set_xlim([date_ini, pd.Timestamp("2012-07-01")])
ax.set_ylim([0, 20])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylabel("Discharge (m³ s$^{-1}$)", fontsize=18)
ax.set_xlabel("Date [-]", fontsize=18)
ax.tick_params(axis="both", labelsize=15)

Question 3:

- Mean error (bias) is 1.509 m3/s
- Biggest deviations are observed during winter months

In [ ]:
mask=~np.isnan(df_sim["Qobs"]) & ~np.isnan(df_sim["Qsim"])
print(f"The mean error is: {np.mean(df_sim[mask]['Qsim']-df_sim[mask]['Qobs'])}")

## Calibrating the hydrological model
<div style="
    width: 100%;
    box-sizing: border-box;
    border: 1px solid #2979ff;
    border-left-width: 5px;
    background-color: #eef5ff;
    padding: 12px 16px;
    margin: 12px 0;
    border-radius: 4px;
">
<strong>Assignment 3</strong><br>

Do a manual calibration by adjusting some of the model parameters for the time period 2012-01-01 2019-12-31. Answer the following questions.

1. During the winter months, $S(t)$ is typically high, and might reach $S_{max}$. Will $P_{exc}$ then, on average, be higher or lower during the winter months (yes or no)? 

2. In case $S(t) = S_{max}$, $P_2 = \alpha P_{exc}$. Based on this knowledge, do you think the default value for $\alpha$ is too high for our catchment? Relate your answer to the above question, and to your answer (figure) to question 1 of assignment 2. Answer in max 5 sentences.


3. Adjust the default value of the parameter $\alpha$ to $0.2$. Remake the figure of simulated vs. observed discharges. During which periods/months do you see the biggest changes?

4. Which type of discharge is still too high after adjusting $\alpha$? $Q_1$ or $Q_2$? Why (max 2 sentences)?. Adjust $Q_{p_{max}}$ to 5, and keep $\alpha = 0.2$. Which of the discharge outputs ($Q_1$ or $Q_2$) does this variable influence? How does it then affect the simulated discharges (max 5 sentences)? Return a figure of the simulated discharges as well.
</div>

Question 1:

$P_{exc}$ is typically higher during the winter months as $P_{in}$ will be lower given the higher values of $S(t)$. You can see this from the formula: $P_{in} = (1 - \frac{S}{S_{max}})^b P$ and $P_{exc}  = P - P_{in}$. 

Question 2:

$\alpha$ is too high. During winter months $P_2$ will be ~ $\alpha P_{exc}$ given the high values of $S(t)$, and $P_{exc}$ will also be higher given that less water can infiltrate in the soil reservoir. You can see this in the produced discharge, which is most off during winter months. So based on this knowledge, too much of the precipitation goes to the fast reacting reservoir, and can be reduced by putting $\alpha$ to a lower value

Question 3: 

In [ ]:
# --- adjust alpha and rerun ---
parameters=Parameters(alpha=0.2)
states=States()
date_ini=pd.Timestamp("2012-01-01")
date_end=pd.Timestamp("2019-12-31")

df_sim=runModelTimePeriod(
    df_forcings=df_forcings,
    parameters=parameters, 
    states=states, 
    date_ini=date_ini, 
    date_end=date_end
    )

# --- plot the observed and simulated discharge ---
PlotQobsSim(
    df_sim=df_sim,
    )


Question 4:

$Q_{P_{max}}$ influences the amount of percolation from $S(t)$ to $S_1(t)$, and thus influences $Q_1$ given that slow reacting reservoir is a linear reservoir. The slow reacting reservoir influences the base flow, and thus adjusts the height (minimal observed values) of the simulated discharge.

In [ ]:
# --- adjust Q_p_max and rerun ---
parameters=Parameters(alpha=0.2, Q_p_max=5)
states=States()
date_ini=pd.Timestamp("2012-01-01")
date_end=pd.Timestamp("2019-12-31")

df_sim=runModelTimePeriod(
    df_forcings=df_forcings,
    parameters=parameters, 
    states=states, 
    date_ini=date_ini, 
    date_end=date_end
    )

# --- plot the observed and simulated discharge ---
PlotQobsSim(
    df_sim=df_sim,
    )

<div style="
    width: 100%;
    box-sizing: border-box;
    border: 1px solid #2979ff;
    border-left-width: 5px;
    background-color: #eef5ff;
    padding: 12px 16px;
    margin: 12px 0;
    border-radius: 4px;
">
<strong>Assignment 4</strong><br>

Calibrate the parameters of the FLEX model using the [Nelder-Mead method](https://en.wikipedia.org/wiki/Nelder%E2%80%93Mead_method). Use the time period 2010-01-01 2013-12-31 for calibration, and the remaining period for validation. 

1. Implement the Nash-Sutcliffe Efficiency (NSE) as the objective function to minimize. What is the optimal value for the NSE? What does a negative NSE mean? Make this clear by transforming the formula of the NSE.


2. Use the `Calibration` function to calibrate the model with the NSE as objective function, with a maximum of 1000 iterations. Use again the 50 warm-up days, and use the default states and parameters. What is the bias for the validation period?

3. Implement the Kling Gupta Efficiency (KGE) similarly as the NSE. Recalibrate your model with the KGE. Run both models from 2014-01-01 onwards with a warm-up period of 50 days. Make a plot of the observed discharges and simulated ones (both NSE and KGE) for the time period 2018-01-01 2019-12-31. 

4. Which calibration (NSE or KGE) performs best in terms of bias and correlation for the validation period? Return the metrics for both calibrations.


</div>

<div class="alert alert-block alert-info"> <b>Tip!! </b> 

- You can use the function `NelderMeadCalibration` to make a model simulation for a specific time period. You can import the function from: `Scripts.scripts_hydrological_model.calibration`.  
- You can compute the correlation between two columns of a dataframe as follows: `df["col1"].corr(df["Col2"])`

</div>

Question 1: 
- Optimal value = 1. 
- Negative values: means that the model run is worse (the RMSE between the model run and the observed discharges) than using the average observations

In [ ]:
def NSE(Qsim: np.ndarray, 
        Qobs: np.ndarray,
        greater_is_better: bool = True
    ) -> float:
    '''Implementation of the Nash-Sutcliffe Efficiency (NSE) as a loss function to evaluate during the calibration of the model parameters.'''

    # implement the NSE
    mask=np.isfinite(Qsim) & np.isfinite(Qobs)
    Qsim = Qsim[mask]
    Qobs = Qobs[mask]

    ss_res = np.sum((Qsim - Qobs) ** 2)
    ss_tot = np.sum((Qobs - np.mean(Qobs)) ** 2)
    loss = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0

    if Qobs.size == 0:
        return np.nan, greater_is_better

    return loss, greater_is_better

Question 2: 

- the bias for the validation period is -0.06203 m3/s

In [ ]:
# --- calibrate the model parameters using the NSE as a loss function ---
params_calib_NSE, history_calib_NSE = NelderMeadCalibration(
    df_forcings=df_forcings,
    loss_function=NSE,
    start_parameters=Parameters(),
    states=States(),
    date_ini=pd.Timestamp("2010-01-01"),
    date_end=pd.Timestamp("2013-12-31"),
    n_warmup_days=50,
    max_iterations=1000,
    x_tolerance=1e-6,
    objective_tolerance=1e-4,
    verbose=True
)

In [ ]:
# --- simulate with the calibrated parameters ---
df_sim_NSE=runModelTimePeriod(
    df_forcings=df_forcings,
    parameters=params_calib_NSE,
    states=States(),
    date_ini=pd.Timestamp("2014-01-01"),
    date_end=df_forcings.index.max(),
)

# --- compute the bias between the observed and simulated discharge ---
mask=~np.isnan(df_sim_NSE["Qobs"]) & ~np.isnan(df_sim_NSE["Qsim"])
print(f"The mean error after NSE calibration is: {np.mean(df_sim_NSE[mask]['Qsim']-df_sim_NSE[mask]['Qobs'])}")

Question 3

In [ ]:
def KGE(Qsim: np.ndarray, 
        Qobs: np.ndarray,
        greater_is_better: bool = True
    ) -> float:
    '''Implementation of the Kling-Gupta Efficiency (KGE) as a loss function to evaluate during the calibration of the model parameters.'''

    # implement the KGE
    mask=np.isfinite(Qsim) & np.isfinite(Qobs)
    Qsim = Qsim[mask]
    Qobs = Qobs[mask]

    if len(Qsim) < 2:
        return np.nan

    r = np.corrcoef(Qsim, Qobs)[0, 1]
    alpha = np.std(Qsim) / np.std(Qobs)
    beta = np.mean(Qsim) / np.mean(Qobs)

    loss=1.0 - np.sqrt(
        (r - 1.0) ** 2
        + (alpha - 1.0) ** 2
        + (beta - 1.0) ** 2
    )

    return loss, greater_is_better

params_calib_KGE, history_calib_KGE = NelderMeadCalibration(
    df_forcings=df_forcings,
    loss_function=KGE,
    start_parameters=Parameters(),
    states=States(),
    date_ini=pd.Timestamp("2010-01-01"),
    date_end=pd.Timestamp("2013-12-31"),
    n_warmup_days=50,
    max_iterations=1000,
    x_tolerance=1e-6,
    objective_tolerance=1e-4,
    verbose=False
)

In [ ]:
# --- simulate with the KGE and compare with the NSE calibration ---
df_sim_KGE=runModelTimePeriod(
    df_forcings=df_forcings,
    parameters=params_calib_KGE,
    states=States(),
    date_ini=pd.Timestamp("2014-01-01"),
    date_end=df_forcings.index.max(),
)

fig,ax=plt.subplots(figsize=(12, 5))
ax.plot(
    df_sim_KGE.index,
    df_sim_KGE["Qsim"],
    color="#6BD117",
    lw=2,
    alpha=1,
    label="Simulated discharge (KGE)"
)
ax.plot(
    df_sim_NSE.index,
    df_sim_NSE["Qobs"],
    color="#3C81AF",
    lw=2,
    ls="-.",
    alpha=1,
    label="Simulated discharge (NSE)"
)
ax.plot(
    df_sim_NSE.index,
    df_sim_NSE["Qsim"],
    color="#FF6347",
    lw=2,
    ls="--",
    alpha=0.7,
    label="Simulated discharge (Original)"
)

ax.set_xlabel("Date [-]", fontsize=18)
ax.set_ylabel("Discharge (m³/s)", fontsize=18)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=15)
ax.legend(frameon=False, fontsize=15, loc="upper left")
ax.set_xlim([pd.Timestamp("2018-01-01"), pd.Timestamp("2019-12-31")])
ax.set_ylim([0, 12])
plt.show()


Question 4:

The NSE has a better correlation, but the KGE displays a slightly better bias

In [ ]:
# --- metrics ---
# Correlation 
mask=~np.isnan(df_sim_KGE["Qobs"])
r_KGE=df_sim_KGE.loc[mask]["Qobs"].corr(df_sim_KGE.loc[mask]["Qsim"])
r_NSE=df_sim_NSE.loc[mask]["Qobs"].corr(df_sim_NSE.loc[mask]["Qsim"])

print(f"Correlation coefficient (KGE): {r_KGE:.3f}")
print(f"Correlation coefficient (NSE): {r_NSE:.3f}")

# Bias
bias_KGE=np.mean(df_sim_KGE.loc[mask]["Qsim"]-df_sim_KGE.loc[mask]["Qobs"])
bias_NSE=np.mean(df_sim_NSE.loc[mask]["Qsim"]-df_sim_NSE.loc[mask]["Qobs"])
print(f"Bias (KGE): {bias_KGE:.3f}")
print(f"Bias (NSE): {bias_NSE:.3f}")


## Simulate with the calibrated model
<div style="
    width: 100%;
    box-sizing: border-box;
    border: 1px solid #2979ff;
    border-left-width: 5px;
    background-color: #eef5ff;
    padding: 12px 16px;
    margin: 12px 0;
    border-radius: 4px;
">
<strong>Assignment 5</strong><br>

A hydropower company wants to know if it is feasible to build a power plant in the Zwalm catchment. Their plant needs a minimal discharge of 2 m<sup>3</sup>s<sup>-1</sup> for at least 20% of the time. 

1. Create a CDF of the simulated discharges for the validiation period. Plot the discharge on the x-axis and the probability of exceedence on the y-axis. 


2. What is the probability of exceedence for the required discharge of 2 m<sup>3</sup>s<sup>-1</sup>? Will they be able to build a hydropower plant (yes or no)?
</div>

</div>

<div class="alert alert-block alert-info"> <b>Tip!! </b> 

$P(x > X) = 1 - P(x \le X)$

</div>

In [ ]:
# --- function to compute the cumulative distribution function (CDF) of the simulated discharge ---
def ComputeCDF(
        df_sim: pd.DataFrame
    ) -> pd.DataFrame:
    '''Function to compute the cumulative distribution function (CDF) of the simulated discharge.'''
    df_cdf=df_sim.sort_values(by="Qsim")
    unique=df_cdf["Qsim"].unique()

    cdf=[(df_cdf["Qsim"]<=unique[i]).sum()/len(df_cdf) for i in range(len(unique))]

    cdf=pd.DataFrame({"Qsim":unique, "CDF":cdf})

    return cdf

# ---- compute the CDF of the simulated discharge for the NSE calibration ---
min_discharge=6

# compute the CDF of the simulated discharge
CDF = ComputeCDF(df_sim_NSE)

# make a plot of the CDF of the simulated discharge
fig,ax = plt.subplots(figsize=(10, 6))
ax.plot(CDF["Qsim"], 1-CDF["CDF"], color="tab:blue", lw=2, label="Probability of exceedance")
ax.axvline(x=min_discharge, color="tab:orange", lw=2, ls="--", label=f"Discharge threshold: {min_discharge} m³/s")
ax.tick_params(axis="both", labelsize=15)
ax.set_xlabel("Discharge (m³ s$^{-1}$)", fontsize=18)
ax.set_ylabel("Probability of exceedance [-]", fontsize=18)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(fontsize=15, loc="upper right", frameon=False)
fig.tight_layout()
plt.show()

# compute the percentage of time that the discharge is above 2 m3/s
pctge = (1-CDF.iloc[np.argmin(np.abs(CDF["Qsim"]-min_discharge))]["CDF"]).item()*100
print(f"The percentage of time that the discharge is above {min_discharge} m³/s is {pctge:.2f}%")